<a href="https://colab.research.google.com/github/khalilosss11/f1_Singapore_GP_2026_predictor/blob/main/singapore_gp_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Singapore GP:                  CUTOFF_DATE = '2026-10-11', CIRCUIT_HISTORY_NAME = 'Singapore Grand Prix'
CUTOFF_DATE = '2026-10-11'            # race day: only data strictly BEFORE this date is used
CIRCUIT_HISTORY_NAME = 'Singapore Grand Prix'
SNAPSHOT = 'fastf1_2025_2026_snapshot.csv'

In [ ]:
# Uses local CSVs if present, otherwise tries to download them from Kaggle
import os
import pandas as pd
import numpy as np

FILES = ['races.csv', 'results.csv', 'drivers.csv', 'constructors.csv', 'qualifying.csv']
DATA_DIR = '.'
if not all(os.path.exists(f) for f in FILES):
    try:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")
    except Exception as e:
        raise SystemExit(f"Upload {FILES} to this Colab session. Details: {e}")

races = pd.read_csv(f'{DATA_DIR}/races.csv')
results = pd.read_csv(f'{DATA_DIR}/results.csv')
drivers = pd.read_csv(f'{DATA_DIR}/drivers.csv')
constructors = pd.read_csv(f'{DATA_DIR}/constructors.csv')
qualifying = pd.read_csv(f'{DATA_DIR}/qualifying.csv')
print("Loaded from:", DATA_DIR, "| last Kaggle season:", races['year'].max())

Using Colab cache for faster access to the 'formula-1-world-championship-1950-2020' dataset.
Loaded from: /kaggle/input/formula-1-world-championship-1950-2020 | last Kaggle season: 2024


In [ ]:
# Joins the tables and fixes types ('\N' for DNFs becomes NaN)
df = results.merge(races[['raceId','year','round','circuitId','date','name']], on='raceId', how='left')
df = df.merge(drivers[['driverId','code']], on='driverId', how='left')
df = df.merge(constructors[['constructorId','name']].rename(columns={'name':'constructor_name'}), on='constructorId', how='left')
q = qualifying[['raceId','driverId','position']].rename(columns={'position':'quali_position'})
df = df.merge(q, on=['raceId','driverId'], how='left')

df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date','raceId']).reset_index(drop=True)
df['won'] = (df['position'] == 1).astype(int)
print(df.shape)

(26759, 27)


In [ ]:
# The cache lives on Google Drive so fetched data survives restarts
# and the 500 calls/hour API limit isn't burned twice
!pip install fastf1 -q
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/f1_project/fastf1_cache', exist_ok=True)

import fastf1
fastf1.Cache.enable_cache('/content/drive/MyDrive/f1_project/fastf1_cache')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.7/424.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.1 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# Pulls every completed race (and its qualifying) of a season;
# races that haven't happened yet are skipped
def fetch_season(season):
    rows = []
    schedule = fastf1.get_event_schedule(season, include_testing=False)
    for _, event in schedule.iterrows():
        round_num, event_name = event['RoundNumber'], event['EventName']
        try:
            race = fastf1.get_session(season, round_num, 'R')
            race.load(laps=False, telemetry=False, weather=False, messages=False)
        except Exception as e:
            print(f"skip {season} R{round_num} {event_name}: {e}")
            continue
        if race.results is None or race.results.empty:
            continue
        try:
            quali = fastf1.get_session(season, round_num, 'Q')
            quali.load(laps=False, telemetry=False, weather=False, messages=False)
            quali_pos = quali.results.set_index('Abbreviation')['Position'].to_dict()
        except Exception:
            quali_pos = {}
        for _, r in race.results.iterrows():
            code = r['Abbreviation']
            rows.append({
                'year': season, 'round': round_num, 'name': event_name,
                'date': event['EventDate'], 'code': code,
                'constructor_name': r['TeamName'], 'grid': r['GridPosition'],
                'position': r['Position'], 'quali_position': quali_pos.get(code),
                'points': r['Points'],
            })
    return pd.DataFrame(rows)

In [ ]:
# First run: fetch from FastF1 and save a snapshot. Later runs read the file.
if os.path.exists(SNAPSHOT):
    new_rows = pd.read_csv(SNAPSHOT)
else:
    new_rows = pd.concat([fetch_season(2025), fetch_season(2026)], ignore_index=True)
    new_rows.to_csv(SNAPSHOT, index=False)

# Keep only races before race day, so the prediction can't see its own result or anything later
new_rows['date'] = pd.to_datetime(new_rows['date'])
new_rows = new_rows[new_rows['date'] < pd.Timestamp(CUTOFF_DATE)].copy()

print(new_rows.shape)
print("Last race used:", new_rows.sort_values('date').iloc[-1][['name', 'date']].to_dict())
assert new_rows['date'].max() < pd.Timestamp(CUTOFF_DATE)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
           

(831, 10)
Last race used: {'name': 'Bahrain Grand Prix', 'date': Timestamp('2026-10-04 00:00:00')}


In [ ]:
# Stacks Kaggle and FastF1 using shared text keys (code, constructor_name, name)
base_cols = ['year','round','name','date','code','constructor_name','grid','position','quali_position','points']
df_combined = pd.concat([df[base_cols], new_rows[base_cols]], ignore_index=True)
df_combined['date'] = pd.to_datetime(df_combined['date'])
df_combined['position'] = pd.to_numeric(df_combined['position'], errors='coerce')
df_combined = df_combined.sort_values('date').reset_index(drop=True)
df_combined['won'] = (df_combined['position'] == 1).astype(int)
print(df_combined.shape)

(27590, 11)


In [ ]:
# Recent form, circuit history, championship standing;
# shift(1) stops a race from seeing its own result (no leakage)
WINDOW = 5
g = df_combined.groupby

df_combined['driver_recent_form'] = g('code')['position'].transform(
    lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
df_combined['constructor_recent_form'] = g('constructor_name')['position'].transform(
    lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
df_combined['driver_circuit_history'] = g(['code','name'])['position'].transform(
    lambda s: s.shift(1).expanding().mean())
df_combined['constructor_circuit_history'] = g(['constructor_name','name'])['position'].transform(
    lambda s: s.shift(1).expanding().mean())
df_combined['champ_points_entering'] = g(['year','code'])['points'].transform(
    lambda s: s.shift(1).cumsum())
df_combined['champ_position_entering'] = g('year')['champ_points_entering'].rank(
    ascending=False, method='min')

In [ ]:
# Keeps only the columns the model needs
feature_cols = ['year','round','name','date','code','constructor_name',
                'grid','quali_position','driver_recent_form','constructor_recent_form',
                'driver_circuit_history','constructor_circuit_history',
                'champ_points_entering','champ_position_entering','position','won']
training_table = df_combined[feature_cols].dropna(subset=['grid']).copy()
print(training_table.shape)

(27590, 16)


In [ ]:
# 2022-2024 are held out as an honest test set; calibrated XGBoost classifier
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV

FEATURES = ['grid','quali_position','driver_recent_form','constructor_recent_form',
            'driver_circuit_history','constructor_circuit_history',
            'champ_points_entering','champ_position_entering']
TEST_SEASONS = [2022, 2023, 2024]

train = training_table[~training_table['year'].isin(TEST_SEASONS)].dropna(subset=FEATURES + ['won'])
test = training_table[training_table['year'].isin(TEST_SEASONS)].dropna(subset=FEATURES + ['won'])
print(f"Train: {len(train)} rows | Test: {len(test)} rows")

X_train, y_train = train[FEATURES], train['won']
X_test, y_test = test[FEATURES], test['won']

base_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss', random_state=42)
model = CalibratedClassifierCV(base_model, method='sigmoid', cv=5)
model.fit(X_train, y_train)
print("Model trained.")

Train: 6364 rows | Test: 1049 rows
Model trained.


In [ ]:
# How often does the model's favourite actually win?
def top1_accuracy(df_, prob_col='win_prob'):
    hits = [int(g_.loc[g_[prob_col].idxmax(), 'won'] == 1) for _, g_ in df_.groupby(['year', 'round'])]
    return sum(hits) / len(hits)

test = test.copy()
test['win_prob_raw'] = model.predict_proba(X_test)[:, 1]
test['win_prob'] = test.groupby(['year','round'])['win_prob_raw'].transform(lambda s: s / s.sum())
print(f"Top-1 accuracy on {TEST_SEASONS}: {top1_accuracy(test):.3f}")

Top-1 accuracy on [2022, 2023, 2024]: 0.571


In [ ]:
# Circuit history comes from every past "Singapore Grand Prix"
# (Kaggle 2008-2024 plus FastF1 2025), so no proxy circuit is needed
rows_2026 = df_combined[df_combined['year'] == 2026]
latest_round = rows_2026['round'].max()
pred = (rows_2026[rows_2026['round'] == latest_round][['code','constructor_name']]
        .drop_duplicates().reset_index(drop=True))

recent = df_combined.sort_values('date')
driver_form = recent.groupby('code')['position'].apply(lambda s: s.tail(5).mean()).rename('driver_recent_form')
team_form = recent.groupby('constructor_name')['position'].apply(lambda s: s.tail(5).mean()).rename('constructor_recent_form')

singapore = df_combined[df_combined['name'] == CIRCUIT_HISTORY_NAME]
driver_circuit = singapore.groupby('code')['position'].mean().rename('driver_circuit_history')
team_circuit = singapore.groupby('constructor_name')['position'].mean().rename('constructor_circuit_history')

champ = rows_2026.groupby('code')['points'].sum().rename('champ_points_entering')

pred = (pred.merge(driver_form, on='code', how='left')
            .merge(team_form, on='constructor_name', how='left')
            .merge(driver_circuit, on='code', how='left')
            .merge(team_circuit, on='constructor_name', how='left')
            .merge(champ, on='code', how='left'))

# Rookies and new teams (Audi, Cadillac) fall back to team history, then the field median
pred['driver_circuit_history'] = pred['driver_circuit_history'].fillna(pred['constructor_circuit_history'])
pred['driver_circuit_history'] = pred['driver_circuit_history'].fillna(pred['driver_circuit_history'].median())
pred['constructor_circuit_history'] = pred['constructor_circuit_history'].fillna(pred['driver_circuit_history'])
pred['champ_position_entering'] = pred['champ_points_entering'].rank(ascending=False, method='min')
print(pred.shape, "(should be 22 rows)")

(22, 8) (should be 22 rows)


In [ ]:
# Main qualifying, Saturday 10 Oct 2026 (not the sprint).
# Russell qualified 6th but starts at the back after a grid penalty.
# Check the official final grid before the race in case it changes.

# Raw qualifying classification, P1 to P22
quali_order = ['VER','LEC','HAM','ANT','NOR','RUS','PIA','GAS','LAW','LIN','HUL',
               'BEA','OCO','SAI','BOR','COL','ALO','ALB','STR','PER','BOT','HAD']

# Starting grid after penalties: Russell drops to the back, everyone behind him moves up one
grid_order = ['VER','LEC','HAM','ANT','NOR','PIA','GAS','LAW','LIN','HUL','BEA',
              'OCO','SAI','BOR','COL','ALO','ALB','STR','PER','BOT','HAD','RUS']

assert len(quali_order) == 22 and len(set(quali_order)) == 22, "Enter all 22 codes exactly once"
assert set(grid_order) == set(quali_order) and len(grid_order) == 22, "grid_order must hold the same 22 codes"

quali_results = pd.DataFrame({'code': quali_order, 'quali_position': range(1, 23)})
quali_results['grid'] = quali_results['code'].map({c: i + 1 for i, c in enumerate(grid_order)})

# errors='ignore' makes this cell safe to re-run
pred = pred.drop(columns=['grid','quali_position'], errors='ignore').merge(quali_results, on='code', how='left')

for col in FEATURES:
    pred[col] = pred[col].fillna(pred[col].median())
assert pred['grid'].notna().all() and len(pred) == 22

In [ ]:
# Win probabilities, normalised so the field sums to 100%
raw = model.predict_proba(pred[FEATURES])[:, 1]
pred['win_probability'] = (raw / raw.sum() * 100).round(1)

result = pred[['code','constructor_name','grid','win_probability']].sort_values('win_probability', ascending=False)
result

,code,constructor_name,grid,win_probability
18,VER,Red Bull,1,57.5
15,LEC,Ferrari,2,13.8
17,ANT,Mercedes,4,5.0
16,HAM,Ferrari,3,1.6
14,HAD,Red Bull,21,1.5
10,PIA,McLaren,6,1.5
11,NOR,McLaren,5,1.4
4,SAI,Williams,13,1.2
5,BOR,Audi,14,1.2
2,OCO,Haas F1 Team,12,1.2


In [ ]:
# One group per race (qid), the model needs rows sorted by group
rank_df = training_table.dropna(subset=FEATURES).copy()
rank_df['qid'] = rank_df.groupby(['year', 'round']).ngroup()
rank_df = rank_df.sort_values('qid').reset_index(drop=True)

# DNFs (position is NaN) get a place just behind the last classified driver
field_max = rank_df.groupby('qid')['position'].transform('max')
rank_df['finish_pos'] = rank_df['position'].fillna(field_max + 1)
rank_df = rank_df.dropna(subset=['finish_pos'])

# Relevance: higher = better finish (winner gets the highest value)
rank_df['relevance'] = (rank_df.groupby('qid')['finish_pos'].transform('max')
                        - rank_df['finish_pos']).astype(int)
print(rank_df.shape)

(7410, 19)


In [ ]:
# Learns pairwise "driver A finishes ahead of driver B" inside each race
from xgboost import XGBRanker

train_r = rank_df[~rank_df['year'].isin(TEST_SEASONS)]
test_r = rank_df[rank_df['year'].isin(TEST_SEASONS)].copy()

ranker = XGBRanker(
    objective='rank:pairwise',
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42)
ranker.fit(train_r[FEATURES], train_r['relevance'], qid=train_r['qid'])
print("Ranker trained.")

Ranker trained.


In [ ]:
# Compares the model with the simplest possible order: everyone finishes where they started
from scipy.stats import spearmanr

def evaluate(df_, score_col):
    rows = []
    for _, g in df_.groupby('qid'):
        g = g.sort_values(score_col, ascending=False)
        pred_top3 = list(g['code'].head(3))
        actual_top3 = list(g.sort_values('finish_pos')['code'].head(3))
        rows.append({
            'winner_correct': pred_top3[0] == actual_top3[0],
            'podium_drivers_correct (of 3)': len(set(pred_top3) & set(actual_top3)),
            'exact_podium_order': pred_top3 == actual_top3,
            'spearman (whole field)': spearmanr(g[score_col], -g['finish_pos'])[0],
        })
    return pd.DataFrame(rows).mean().round(3)

test_r['model_score'] = ranker.predict(test_r[FEATURES])
test_r['grid_score'] = -test_r['grid']

print("Ranker:\n", evaluate(test_r, 'model_score'))
print("\nBaseline (grid order):\n", evaluate(test_r, 'grid_score'))

Ranker:
 winner_correct                   0.540
podium_drivers_correct (of 3)    1.984
exact_podium_order               0.095
spearman (whole field)           0.696
dtype: float64

Baseline (grid order):
 winner_correct                   0.365
podium_drivers_correct (of 3)    1.825
exact_podium_order               0.111
spearman (whole field)           0.556
dtype: float64


In [ ]:
# Higher score = finishes further ahead. Scores are for sorting only, not probabilities.
pred['rank_score'] = ranker.predict(pred[FEATURES])
order = pred.sort_values('rank_score', ascending=False).reset_index(drop=True)
order.index = order.index + 1
order[['code', 'constructor_name', 'grid', 'rank_score']]

,code,constructor_name,grid,rank_score
1,VER,Red Bull,1,1.994432
2,LEC,Ferrari,2,1.481794
3,ANT,Mercedes,4,1.211412
4,NOR,McLaren,5,0.991444
5,HAM,Ferrari,3,0.990133
6,PIA,McLaren,6,0.593520
7,RUS,Mercedes,22,0.468328
8,HUL,Audi,10,0.144662
9,GAS,Alpine F1 Team,7,0.103079
10,HAD,Red Bull,21,0.010947


In [ ]:
# A verifiable pre-race prediction for your portfolio
from datetime import datetime
stamp = datetime.now().strftime('%Y%m%d_%H%M')
result.to_csv(f'/content/drive/MyDrive/f1_project/singapore_gp_2026_prediction_{stamp}.csv', index=False)
print("Saved prediction with timestamp", stamp)

Saved prediction with timestamp 20261011_1411
